In [ ]:
import pandas as pd
import altair as alt
import numpy as np

In [ ]:
data = pd.read_csv("silica_predictions/ucsf_ours_g2m_cn_gmm32_softsoft_manual_override_pred.csv") #silica_predictions/uscf_ours_g2m_cn200_knn_pred.csv") #uscf_ours_g2m_cn_gmm32_softsoft_pred.csv")

#"all_uscf_ours_srh7db_center200_pred.csv")

#all_uscf_ours_center1000_pred.csv")
#"all_uscf_ours_srh7db_center20_pred.csv")
#"all_uscf_our_pred.csv")


data["prediction"] = np.power(data["prediction"], 1)
#data = pd.read_csv("all_uscf_dv2_pred.csv")


In [ ]:
data["tumor_pred"] =  data["prediction"] < 0.5
slide_gb = data.groupby("slide")["tumor_pred"]

In [ ]:
slide_pred = pd.DataFrame({"tumor": slide_gb.sum(),
              "all": slide_gb.count()}).reset_index()
slide_pred["tumor_rate"] =  slide_pred["tumor"] / slide_pred["all"]

In [ ]:
proper_data = pd.concat([
    pd.read_csv("db_srhdg/db_srhdg_prosp_srhwithgt.csv", dtype=str),
    pd.read_csv("db_srhdg/db_srhdg_retro_v3.1_mp_allpatch_srhwithgt.csv", dtype=str)
])
proper_data["slide"] = proper_data.apply(lambda x: "-".join([x["patient"], x["mosaic"]]), axis=1)
proper_data = proper_data[["patient", "mosaic", "slide", "nio"]].rename({"nio": "melike_score"}, axis=1).reset_index()


In [ ]:
melike_raw = pd.read_csv("db_srhdg/SRHcases_ForMelike-MP-2025-03-15.csv", dtype=str)
melike_raw["slide"] = melike_raw.apply(lambda x: "-".join([x["patient_id"], x["slide_id"]]), axis=1)
melike_raw = melike_raw[["patient_id", "slide_id", "slide", "Melike NIO score"]].rename({"Melike NIO score": "melike_score"}, axis=1).reset_index()


In [ ]:
melike_raw.set_index('slide', inplace=True)
proper_data.set_index('slide', inplace=True)
melike_raw['melike_score'].update(proper_data['melike_score'])
melike_raw.reset_index(inplace=True)

In [ ]:
sample_slide_map = pd.read_csv("db_srhdg/SRHcases_ForMelike-MP-2025-03-15.csv", dtype=str)
sample_slide_map = pd.DataFrame({
    "slide_id": sample_slide_map["slide_id"],
    "pbs": sample_slide_map["patient_id"] + "-" + sample_slide_map["barcode"] + "-" + sample_slide_map["sample_id"]})

fg_score_ucsf = pd.read_csv("db_srhdg/scoresforsanjeev_ucsf.csv").dropna()
fg_score_ucsf["pbs"] = "NIO_UCSF_" + fg_score_ucsf["patient_num"].astype(str) + "-" + fg_score_ucsf["barcode"] +  "-" + fg_score_ucsf["sample_id"]
fg_score_ucsf = fg_score_ucsf.merge(sample_slide_map, on="pbs", how="left")
fg_score_ucsf = fg_score_ucsf.rename({"slide_id": "mosaic"}, axis=1)
fg_score_ucsf = fg_score_ucsf.drop("pbs", axis=1).dropna()
fg_score_ucsf["inst"] = "UCSF"

fg_score_ukk = pd.read_csv("db_srhdg/scoresforsanjeev_ukk.csv")
fg_score_ukk["inst"] = "UKK"

fg_score = pd.concat([fg_score_ucsf, fg_score_ukk]).drop(["sample_id", "barcode"], axis=1)

fg_score["slide"] = "NIO_" + fg_score["inst"] + "_" + fg_score["patient_num"].astype(str) + "-" + fg_score["mosaic"].astype(str)

In [ ]:
fg_meta = pd.merge(melike_raw, fg_score.dropna(), on="slide", how="outer")

In [ ]:
#fg_meta.to_csv("db_srhdg/fg_meta.csv", index=False)

In [ ]:
slide_pred = slide_pred.merge(fg_meta[["slide", "melike_score", "fullsrh", "fastsrh"]], on="slide", how="left")

In [ ]:
slide_pred_filt = slide_pred[["slide", "tumor_rate", "melike_score", "all","fullsrh"]].dropna()
slide_pred_filt = slide_pred_filt[slide_pred_filt["melike_score"].str.isnumeric()]

In [ ]:
len(slide_pred_filt[(slide_pred_filt["tumor_rate"] >.9) & (slide_pred_filt["fullsrh"] > 0.9)]["slide"].tolist())

In [ ]:
alt.data_transformers.disable_max_rows()

tsne_unit_axis = alt.Axis(tickSize=0,
                          values=np.linspace(0,1,6),
                          domain=False,
                          labels=False, title=""
                          )

base_chart = alt.Chart(
    slide_pred_filt,
)

both = base_chart.mark_point(filled=True).encode(
    x=alt.X("fullsrh:Q",axis=tsne_unit_axis),
    y=alt.Y('tumor_rate:Q',axis=tsne_unit_axis),
    tooltip=["slide", "all"],
    color="melike_score"
).properties(width=512, height=512)

tumor_rate = base_chart.mark_tick().encode(
    x=alt.X("melike_score",axis=alt.Axis(tickWidth=0)),
    y=alt.Y('tumor_rate:Q',axis=alt.Axis(tickSize=0,
                          values=np.linspace(0,1,6))),
    tooltip=["slide", "all"],
    color="melike_score"
).properties(height=512)

fastsrh = base_chart.mark_tick().encode(
    x=alt.X('fullsrh:Q',axis=alt.Axis(tickSize=0,
                          values=np.linspace(0,1,6))),
    y=alt.Y("melike_score",axis=alt.Axis(tickWidth=0)),
    tooltip=["slide", "all"],
    color="melike_score"
).properties(width=512)

(tumor_rate | (both & fastsrh)).configure_axis(
    labelFontSize=14,
    titleFontSize=14,
).configure_legend(titleFontSize=14).interactive()

In [ ]:
(tumor_rate | (both & fastsrh)).configure_axis(
    labelFontSize=16,
    titleFontSize=16,
).configure_legend(titleFontSize=16).interactive().save("silica_predictions/melike_stats/gmm32_softsoft_manual_override.png")

In [ ]:
alt.data_transformers.disable_max_rows()

tsne_unit_axis = alt.Axis(tickSize=0,
                          values=np.linspace(0,1,6),
                          domain=False,
                          labels=False, title=""
                          )

base_chart = alt.Chart(
    slide_pred_filt,
)

both = base_chart.mark_point(filled=True).encode(
    x=alt.X("fullsrh:Q",axis=tsne_unit_axis),
    y=alt.Y('tumor_rate:Q',axis=tsne_unit_axis),
    tooltip=["slide", "all"],
    color="melike_score"
).properties(width=512, height=512)

tumor_rate = base_chart.mark_tick().encode(
    x=alt.X("melike_score",axis=alt.Axis(tickWidth=0)),
    y=alt.Y('tumor_rate:Q',axis=alt.Axis(tickSize=0,
                          values=np.linspace(0,1,6))),
    tooltip=["slide", "all"],
    color="melike_score"
).properties(height=512)

fastsrh = base_chart.mark_tick().encode(
    x=alt.X('fullsrh:Q',axis=alt.Axis(tickSize=0,
                          values=np.linspace(0,1,6))),
    y=alt.Y("melike_score",axis=alt.Axis(tickWidth=0)),
    tooltip=["slide", "all"],
    color="melike_score"
).properties(width=512)

(tumor_rate | (both & fastsrh)).interactive()

In [ ]:
slide_pred_melike

In [ ]:
slide_pred_filt[(slide_pred_filt["tumor_rate"]<0.1) & (slide_pred_filt["fullsrh"]>0.9)]["slide"].tolist()

In [ ]:
slide_pred_filt[slide_pred_filt["slide"]=="NIO_UCSF_197741-1"]

# compute correlation

In [ ]:
def proc_prediction(data):
    data["tumor_pred"] =  data["prediction"] < 0.5
    slide_gb = data.groupby("slide")["tumor_pred"]
    slide_pred = pd.DataFrame({"tumor": slide_gb.sum(),
              "all": slide_gb.count()}).reset_index()
    slide_pred["tumor_rate"] =  slide_pred["tumor"] / slide_pred["all"]
    
    return slide_pred

In [ ]:
data_gmm = pd.read_csv("silica_predictions/uscf_ours_g2m_cn_gmm32_softsoft_pred.csv")
data_knn = pd.read_csv("silica_predictions/uscf_ours_g2m_cn200_knn_pred.csv")
data_gmm_ours = pd.read_csv("silica_predictions/ucsf_ours_g2m_cn_gmm32_softsoft_manual_override_pred.csv")

In [ ]:
gmm_pred =  proc_prediction(data_gmm)
knn_pred = proc_prediction(data_knn)
gmm_pred_ours =  proc_prediction(data_gmm_ours)

In [ ]:
diff = pd.DataFrame({"slide": knn_pred["slide"],
              "diff": (gmm_pred["tumor"] - knn_pred["tumor"]).abs(), "all": gmm_pred["all"]})
diff["diff_rate"] = diff["diff"] / diff["all"]

In [ ]:
diff

In [ ]:
diff[(diff["diff_rate"]>0.2) | (diff["diff"]>500)]["slide"]

In [ ]:
glob(f"/nfs/turbo/umms-tocho/root_srh_db/UCSF/NIO_UCSF_559/2/mosaic*/img*_1*.dcm")

In [ ]:
glob(f"/nfs/turbo/umms-tocho/root_srh_db/UKK/NIO_UKK_267/1/mosaic*/IMG00001.dcm")

In [ ]:
slide_pred_filt

In [ ]:
from scipy.stats import pearsonr
from scipy.stats import spearmanr

corr, p = pearsonr(slide_pred_filt["tumor_rate"], slide_pred_filt["melike_score"].astype(int))

In [ ]:
corr

In [ ]:
pearsonr(slide_pred_filt["tumor_rate"], slide_pred_filt["melike_score"].astype(int))

In [ ]:

corr, p = pearsonr(slide_pred_filt["fullsrh"], slide_pred_filt["melike_score"].astype(int))
corr